# Titanic Veri Seti ile Tahmin

Bu not defteri, `titanic.csv` dosyasındaki yolcuları kullanarak hayatta kalma tahmini yapan bir makine öğrenmesi modeli kurar. Eksik değerler doldurulur, kategorik sütunlar kodlanır ve model bir test kümesi üzerinde değerlendirilir.

In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Veriyi yükle
df = pd.read_csv('titanic.csv')

# Tahminde kullanacağımız sütunlar
feature_columns = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target_column = 'Survived'

# Eksik değerleri otomatik işleyen veri seti
X = df[feature_columns].copy()
y = df[target_column].copy()

numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Sex', 'Embarked']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Dogruluk:', accuracy_score(y_test, y_pred))
print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred))

Dogruluk: 0.8044692737430168

Confusion matrix:
[[98 12]
 [23 46]]

Classification report:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



In [6]:
sample_passenger = pd.DataFrame([
    {
        'Pclass': 1,
        'Sex': 'female',
        'Age': 22,
        'SibSp': 1,
        'Parch': 0,
        'Fare': 7.25,
        'Embarked': 'S',
    }
])

predicted_class = model.predict(sample_passenger)[0]
predicted_probability = model.predict_proba(sample_passenger)[0][1]

print('Ornek yolcu tahmini:')
print('Hayatta kalir' if predicted_class == 1 else 'Hayatta kalmaz')
print(f'Hayatta kalma olasiligi: {predicted_probability:.3f}')

Ornek yolcu tahmini:
Hayatta kalir
Hayatta kalma olasiligi: 0.924
